# Re-embed Corpus with Modern Embedding Model

Replaces the existing MiniLM-L6-v2 (384-dim, 22M params, 2021) embeddings
with a modern model. Run on Colab with A100/L4 for speed.

**Candidate models (pick one):**
- `BAAI/bge-base-en-v1.5` — 110M params, 768-dim, strong MTEB scores
- `BAAI/bge-large-en-v1.5` — 335M params, 1024-dim, best quality
- `nomic-ai/nomic-embed-text-v1.5` — 137M params, 768-dim, good MPS support
- `thenlper/gte-base` — 110M params, 768-dim
- `NeuML/pubmedbert-base-embeddings` — biomedical domain, may outperform on clinical text

**Steps:**
1. Load doc_texts from HuggingFace Hub
2. Encode all ~384k docs with the new model
3. Evaluate on TREC/KZ with embedding-only and hybrid retrieval
4. Compare against MiniLM baseline
5. Optionally push new embeddings to HuggingFace Hub

In [ ]:
# Option 1: If running locally alongside the repo
import sys; sys.path.insert(0, "../src")

# Option 2: In Colab, clone and add to path
# !git clone https://github.com/semajyllek/ctmatch.git
# import sys; sys.path.insert(0, "/content/ctmatch/src")

# Dependencies
!pip install -q sentence-transformers datasets scikit-learn transformers tqdm evaluate lxml rank-bm25


In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
elif torch.backends.mps.is_available():
    print('MPS available (Apple Silicon)')
else:
    print('CPU only — this will be slow for 384k docs')

In [ ]:
import os
import numpy as np
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

## Choose embedding model

In [ ]:
# CHANGE THIS to try different models
NEW_MODEL = 'BAAI/bge-base-en-v1.5'
OLD_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

print(f'New model: {NEW_MODEL}')
new_encoder = SentenceTransformer(NEW_MODEL)
old_encoder = SentenceTransformer(OLD_MODEL)
print(f'New embedding dim: {new_encoder.get_sentence_embedding_dimension()}')
print(f'Old embedding dim: {old_encoder.get_sentence_embedding_dimension()}')

## Load document texts from HuggingFace Hub

In [ ]:
HF_ROOT = 'semaj83/ctmatch_ir'

print('Loading doc texts...')
ds = load_dataset(HF_ROOT, data_files='doc_texts.txt')
doc_texts = [row['text'] for row in ds['train']]
print(f'Loaded {len(doc_texts)} documents')
print(f'Sample: {doc_texts[0][:200]}...')

## Re-embed with new model

In [ ]:
BATCH_SIZE = 256

print(f'Encoding {len(doc_texts)} documents with {NEW_MODEL}...')
print(f'Batch size: {BATCH_SIZE}')

new_embeddings = new_encoder.encode(
    doc_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print(f'Shape: {new_embeddings.shape}')
new_emb_df = pd.DataFrame(new_embeddings)

## Load existing MiniLM embeddings for comparison

In [ ]:
print('Loading old MiniLM embeddings...')
old_ds = load_dataset(HF_ROOT, data_files='doc_embeddings.txt')
old_arrays = [np.asarray(a['text'].split(','), dtype=float) for a in old_ds['train']]
old_emb_df = pd.DataFrame(old_arrays)
print(f'Old shape: {old_emb_df.shape}')

## Load eval data

In [ ]:
from ctmatch.evaluation.eval_utils import (
    get_trec_topic2text, get_kz_topic2text,
    calc_first_positive_rank, calc_f1,
)

DATA_ROOT = os.environ.get('CTMATCH_DATA_ROOT', '../data')
TREC_TOPIC_PATH = os.path.join(DATA_ROOT, 'trec_data/trec_21_topics.xml')
KZ_TOPIC_PATH = os.path.join(DATA_ROOT, 'kz_data/topics-2014_2015-description.topics')
TREC_REL_PATH = os.path.join(DATA_ROOT, 'trec_data/trec_21_judgments.txt')
KZ_REL_PATH = os.path.join(DATA_ROOT, 'kz_data/qrels-clinical_trials.txt')

topic2text = {}
if os.path.exists(TREC_TOPIC_PATH):
    topic2text.update(get_trec_topic2text(TREC_TOPIC_PATH))
if os.path.exists(KZ_TOPIC_PATH):
    topic2text.update(get_kz_topic2text(KZ_TOPIC_PATH))

rel_dict = {}
for rp in [TREC_REL_PATH, KZ_REL_PATH]:
    if not os.path.exists(rp): continue
    with open(rp) as f:
        for line in f:
            tid, _, did, rel = line.split()
            rel_dict.setdefault(tid, {})[did] = int(rel)

# index2docid
idx_ds = load_dataset(HF_ROOT, data_files='index2docid.txt')
index2docid = pd.DataFrame(idx_ds['train'])

def get_doc_indices(doc_ids):
    return [np.where(index2docid['text'] == d)[0][0]
            for d in doc_ids if len(np.where(index2docid['text'] == d)[0]) > 0]

print(f'{len(topic2text)} topics, {len(rel_dict)} with judgments')

## Evaluate: old vs new embeddings

In [ ]:
from ctmatch.matching.retrieval.embeddings import embedding_only_filter

MAX_TOPICS = 50

def eval_emb_model(encoder, emb_df, name):
    fprs, frrs, f1s = [], [], []
    for topic_id, topic_text in tqdm(list(topic2text.items())[:MAX_TOPICS], desc=name):
        if topic_id not in rel_dict: continue
        doc_ids = list(rel_dict[topic_id].keys())
        doc_set = get_doc_indices(doc_ids)
        if not doc_set: continue

        topic_emb = encoder.encode([topic_text])[0]
        ranked = embedding_only_filter(topic_emb, emb_df, doc_set, top_n=len(doc_set))
        ranked_ids = [index2docid.iloc[i].values[0] for i in ranked]

        fpr, frr = calc_first_positive_rank(ranked_ids, rel_dict[topic_id])
        f1 = calc_f1(ranked_ids, rel_dict[topic_id])
        fprs.append(fpr); frrs.append(frr); f1s.append(f1)

    return {'name': name, 'mean_fpr': np.mean(fprs), 'mean_mrr': np.mean(frrs), 'mean_f1': np.mean(f1s)}

old_res = eval_emb_model(old_encoder, old_emb_df, f'MiniLM-L6-v2 (old)')
new_res = eval_emb_model(new_encoder, new_emb_df, f'{NEW_MODEL} (new)')

df = pd.DataFrame([old_res, new_res]).set_index('name')
df.columns = ['Mean FPR ↓', 'Mean MRR ↑', 'Mean F1 ↑']
df.round(4)

## Evaluate: new embeddings + BM25 hybrid

In [ ]:
from ctmatch.matching.retrieval.bm25 import BM25Index
from ctmatch.matching.retrieval.hybrid import hybrid_filter

bm25_index = BM25Index(doc_texts)

def eval_hybrid(encoder, emb_df, name, rrf_k=60):
    fprs, frrs, f1s = [], [], []
    for topic_id, topic_text in tqdm(list(topic2text.items())[:MAX_TOPICS], desc=name):
        if topic_id not in rel_dict: continue
        doc_ids = list(rel_dict[topic_id].keys())
        doc_set = get_doc_indices(doc_ids)
        if not doc_set: continue

        topic_emb = encoder.encode([topic_text])[0]
        ranked = hybrid_filter(
            bm25_index, topic_text, topic_emb, emb_df,
            doc_set, top_n=len(doc_set), rrf_k=rrf_k,
        )
        ranked_ids = [index2docid.iloc[i].values[0] for i in ranked]

        fpr, frr = calc_first_positive_rank(ranked_ids, rel_dict[topic_id])
        f1 = calc_f1(ranked_ids, rel_dict[topic_id])
        fprs.append(fpr); frrs.append(frr); f1s.append(f1)

    return {'name': name, 'mean_fpr': np.mean(fprs), 'mean_mrr': np.mean(frrs), 'mean_f1': np.mean(f1s)}

hybrid_old = eval_hybrid(old_encoder, old_emb_df, 'Hybrid MiniLM+BM25')
hybrid_new = eval_hybrid(new_encoder, new_emb_df, f'Hybrid {NEW_MODEL}+BM25')

all_results = pd.DataFrame([old_res, new_res, hybrid_old, hybrid_new]).set_index('name')
all_results.columns = ['Mean FPR ↓', 'Mean MRR ↑', 'Mean F1 ↑']
all_results.round(4)

## Save new embeddings (optional)

In [ ]:
# Save locally
SAVE_PATH = f'doc_embeddings_{NEW_MODEL.split("/")[-1]}.npy'
np.save(SAVE_PATH, new_embeddings)
print(f'Saved to {SAVE_PATH} ({os.path.getsize(SAVE_PATH) / 1e6:.1f} MB)')

In [ ]:
# Push to HuggingFace Hub (uncomment to run)
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_file(
#     path_or_fileobj=SAVE_PATH,
#     path_in_repo=f'doc_embeddings_{NEW_MODEL.split("/")[-1]}.npy',
#     repo_id='semaj83/ctmatch_ir',
#     repo_type='dataset',
# )
# print('Pushed to HuggingFace Hub')